# Curriculum QA Verifier

Checks every `.py` lesson file for:
1. New `══` border format (not old `==` format)
2. No `input()` calls
3. No emojis
4. At least 3 EXERCISE/TASK sections
5. At least 3 EXAMPLE/PART blocks (non-project files only)
6. Blank workspace (4+ blank lines after EXERCISE/TASK)
7. Python syntax validity (`ast.parse`)

In [ ]:
import os
import ast
import re

CURRICULUM_DIR = os.path.abspath(os.path.dirname("__file__"))
if not os.path.exists(os.path.join(CURRICULUM_DIR, "README.py")):
    CURRICULUM_DIR = os.path.abspath(".")

SKIP_FILES = {"verify_curriculum.py", "README.py", "W12_OVERVIEW.py"}
EMOJI_PATTERN = re.compile(r"[\U0001F000-\U0010FFFF]")

In [ ]:
def check_file(filepath):
    """Return a list of issues found in a .py lesson file."""
    issues = []
    try:
        with open(filepath, encoding="utf-8") as f:
            source = f.read()
    except Exception as e:
        return [f"could not read file: {e}"]

    # 1. New border format
    if "\u2550\u2550" not in source:
        issues.append("missing border -- file may still use old == format")

    # 2. No input()
    if "input(" in source:
        issues.append("contains input() -- not allowed in lesson files")

    # 3. No emojis
    if EMOJI_PATTERN.search(source):
        issues.append("contains emoji characters -- remove them")

    # 4. At least 3 EXERCISE or TASK markers
    exercise_count = source.count("EXERCISE") + source.count("TASK")
    if exercise_count < 3:
        issues.append(f"only {exercise_count} EXERCISE/TASK section(s) -- need at least 3")

    # 5. At least 3 EXAMPLE or PART markers (skip project/overview files)
    is_project = re.search(r"(_D6_|_D7_|OVERVIEW|_Project_|_project_)", filepath)
    if not is_project:
        example_count = source.count("EXAMPLE") + source.count("PART")
        if example_count < 3:
            issues.append(f"only {example_count} EXAMPLE/PART block(s) -- need at least 3")

    # 6. Student workspace (4+ consecutive blank lines after EXERCISE/TASK)
    exercise_positions = [m.start() for m in re.finditer(r"EXERCISE|TASK", source)]
    workspace_found = False
    for pos in exercise_positions:
        if re.search(r"\n{4,}", source[pos:pos + 2000]):
            workspace_found = True
            break
    if not workspace_found and exercise_count >= 1:
        issues.append("no blank workspace (4+ blank lines) after any EXERCISE/TASK")

    # 7. Python syntax
    try:
        ast.parse(source)
    except SyntaxError as e:
        issues.append(f"SYNTAX ERROR line {e.lineno}: {e.msg}")

    return issues

In [ ]:
def collect_lesson_files():
    """Walk Curriculum/ and return sorted .py lesson file paths."""
    lesson_files = []
    for root, dirs, files in os.walk(CURRICULUM_DIR):
        dirs.sort()
        for filename in sorted(files):
            if not filename.endswith(".py"):
                continue
            if filename in SKIP_FILES or filename.startswith("verify_"):
                continue
            lesson_files.append(os.path.join(root, filename))
    return lesson_files

## Run QA

In [ ]:
lesson_files = collect_lesson_files()
results = {}
for filepath in lesson_files:
    rel = os.path.relpath(filepath, CURRICULUM_DIR)
    results[rel] = check_file(filepath)

total_files = len(results)
total_issues = sum(len(v) for v in results.values())
files_passing = sum(1 for v in results.values() if not v)
files_failing = total_files - files_passing

print("=" * 64)
print("  CURRICULUM QA REPORT (.py files)")
print(f"  Files checked : {total_files}")
print(f"  Passing       : {files_passing}")
print(f"  Failing       : {files_failing}")
print(f"  Total issues  : {total_issues}")
print("=" * 64)
print()

current_week = None
for rel_path, issues in results.items():
    week_folder = rel_path.split(os.sep)[0] if os.sep in rel_path else ""
    if week_folder != current_week:
        current_week = week_folder
        print(f"  {week_folder}")
        print(f"  {'---' * 17}")
    filename = os.path.basename(rel_path)
    status = "PASS" if not issues else "FAIL"
    print(f"    {status}  {filename}")
    for issue in issues:
        print(f"          - {issue}")

print()
print("=" * 64)
if files_failing == 0:
    print("  All files pass. Curriculum QA clean.")
else:
    print(f"  {files_failing} file(s) need attention. Fix issues above and re-run.")
print("=" * 64)